In [1]:
from google.colab import drive
drive.mount('/content/drive',force_remount=True)

Mounted at /content/drive


In [2]:

from pathlib import Path



DATA_DIR = Path("/content/drive/MyDrive/DataScience/data")

# File paths
PATH_2010 = DATA_DIR / "medicaid_data_2010.csv"
PATH_2011 = DATA_DIR / "medicaid_data_2011.csv"
PATH_2012 = DATA_DIR / "medicaid_data_2012.csv"
PATH_2013 = DATA_DIR / "medicaid_data_2013.csv"
PATH_2014 = DATA_DIR / "medicaid_data_2014.csv"
PATH_2015 = DATA_DIR / "medicaid_data_2015.csv"
PATH_2016 = DATA_DIR / "medicaid_data_2016.csv"
PATH_2017 = DATA_DIR / "medicaid_data_2017.csv"
PATH_2018 = DATA_DIR / "medicaid_data_2018.csv"
PATH_2019 = DATA_DIR / "medicaid_data_2019.csv"
PATH_2020 = DATA_DIR / "medicaid_data_2020.csv"
PATH_2021 = DATA_DIR / "medicaid_data_2021.csv"
PATH_2022 = DATA_DIR / "medicaid_data_2022.csv"
PATH_2023 = DATA_DIR / "medicaid_data_2023.csv"
PATH_2024 = DATA_DIR / "medicaid_data_2024.csv"
PATH_2025 = DATA_DIR / "medicaid_data_2025.csv"

In [4]:
# combining

import pandas as pd


paths_to_use = {
    2024: PATH_2024,
    2025: PATH_2025
}

dfs = []

for year, path in paths_to_use.items():
    temp_df = pd.read_csv(path, low_memory=False)
    temp_df["Source_Year"] = year
    dfs.append(temp_df)
    print(f"Loaded {year}: {temp_df.shape}")

df = pd.concat(dfs, ignore_index=True)

print("Combined df shape:", df.shape)
print(df.head())



Loaded 2024: (5284306, 16)
Loaded 2025: (2637009, 16)
Combined df shape: (7921315, 16)
  Utilization Type State      NDC  Labeler Code  Product Code  Package Size  \
0             FFSU    AK  2143380             2          1433            80   
1             FFSU    AK  2143480             2          1434            80   
2             FFSU    AK  2143611             2          1436            11   
3             FFSU    AK  2144509             2          1445             9   
4             FFSU    AK  2144511             2          1445            11   

   Year  Quarter  Suppression Used Product Name  Units Reimbursed  \
0  2024        4             False   TRULICITY              226.0   
1  2024        4             False   TRULICITY              216.0   
2  2024        4             False   EMGALITY P              32.0   
3  2024        4              True   TALTZ AUTO               NaN   
4  2024        4             False   TALTZ AUTO              37.0   

   Number of Prescripti

## CLEANING

In [5]:
# Building id

print("Step 2: Cleaning Medicaid NDC pipeline...")

df = df.copy()

# Clean component columns
df["Labeler Code"] = (
    df["Labeler Code"]
    .astype(str)
    .str.strip()
    .str.replace(".0", "", regex=False)
    .str.replace(r"\D", "", regex=True)
    .str.zfill(5)
)

df["Product Code"] = (
    df["Product Code"]
    .astype(str)
    .str.strip()
    .str.replace(".0", "", regex=False)
    .str.replace(r"\D", "", regex=True)
    .str.zfill(4)
)

df["Package Size"] = (
    df["Package Size"]
    .astype(str)
    .str.strip()
    .str.replace(".0", "", regex=False)
    .str.replace(r"\D", "", regex=True)
    .str.zfill(2)
)

# Clean raw NDC for audit
df["NDC_clean"] = (
    df["NDC"]
    .astype(str)
    .str.strip()
    .str.replace("-", "", regex=False)
    .str.replace(".0", "", regex=False)
    .str.replace(r"\D", "", regex=True)
    .str.zfill(11)
)

# Build standardized IDs from trusted components
df["ProductID"] = df["Labeler Code"] + df["Product Code"]
df["NDC_standardized"] = df["ProductID"] + df["Package Size"]

# Compare raw cleaned NDC vs rebuilt NDC
df["NDC_match_flag"] = df["NDC_clean"] == df["NDC_standardized"]

print("Total rows:", len(df))
print("NDC matches:", df["NDC_match_flag"].sum())
print("NDC mismatches:", (~df["NDC_match_flag"]).sum())

print(df[[
    "NDC",
    "Labeler Code",
    "Product Code",
    "Package Size",
    "ProductID",
    "NDC_clean",
    "NDC_standardized",
    "NDC_match_flag"
]].head(10))

Step 2: Cleaning Medicaid NDC pipeline...
Total rows: 7921315
NDC matches: 7921315
NDC mismatches: 0
       NDC Labeler Code Product Code Package Size  ProductID    NDC_clean  \
0  2143380        00002         1433           80  000021433  00002143380   
1  2143480        00002         1434           80  000021434  00002143480   
2  2143611        00002         1436           11  000021436  00002143611   
3  2144509        00002         1445           09  000021445  00002144509   
4  2144511        00002         1445           11  000021445  00002144511   
5  2144527        00002         1445           27  000021445  00002144527   
6  2145780        00002         1457           80  000021457  00002145780   
7  2146080        00002         1460           80  000021460  00002146080   
8  2147180        00002         1471           80  000021471  00002147180   
9  2148480        00002         1484           80  000021484  00002148480   

  NDC_standardized  NDC_match_flag  
0      0000214

## Building New Df for Product

In [6]:
print("Step 3: Creating FDA feature dataframe...")

PATH_TO_PRODUCT_TXT = "/content/product.txt"

fda_df = pd.read_csv(
    PATH_TO_PRODUCT_TXT,
    sep="\t",
    encoding="latin1",
    low_memory=False
)

# Keep valid PRODUCTNDC rows
fda_df = fda_df[fda_df["PRODUCTNDC"].notna()].copy()

fda_df["PRODUCTNDC"] = (
    fda_df["PRODUCTNDC"]
    .astype(str)
    .str.strip()
)

fda_df = fda_df[fda_df["PRODUCTNDC"].str.contains("-", na=False)].copy()

# Split PRODUCTNDC
split_ndc = fda_df["PRODUCTNDC"].str.split("-", expand=True)

# Build FDA ProductID
fda_df["FDA_ProductID"] = (
    split_ndc[0].str.zfill(5) +
    split_ndc[1].str.zfill(4)
)

# Extract features
fda_df["Therapeutic_Class"] = (
    fda_df["PHARM_CLASSES"]
    .astype(str)
    .str.split(",")
    .str[0]
    .str.replace(r"\[.*?\]", "", regex=True)
    .str.strip()
)

fda_df["EPC_Class"] = (
    fda_df["PHARM_CLASSES"]
    .astype(str)
    .str.extract(r"([^,\[]+)\s*\[EPC\]", expand=False)
)

fda_df["MoA_Class"] = (
    fda_df["PHARM_CLASSES"]
    .astype(str)
    .str.extract(r"([^,\[]+)\s*\[MoA\]", expand=False)
)

fda_df["DOSAGEFORMNAME"] = fda_df["DOSAGEFORMNAME"].astype(str).str.strip()
fda_df["ROUTENAME"] = fda_df["ROUTENAME"].astype(str).str.strip()
fda_df["LABELERNAME"] = fda_df["LABELERNAME"].astype(str).str.strip()
fda_df["MARKETINGCATEGORYNAME"] = fda_df["MARKETINGCATEGORYNAME"].astype(str).str.strip()

fda_df["Strength"] = pd.to_numeric(
    fda_df["ACTIVE_NUMERATOR_STRENGTH"],
    errors="coerce"
)

fda_df["ACTIVE_INGRED_UNIT"] = fda_df["ACTIVE_INGRED_UNIT"].astype(str).str.strip()

fda_df["Start_Marketing_Year"] = pd.to_numeric(
    fda_df["STARTMARKETINGDATE"].astype(str).str[:4],
    errors="coerce"
)

fda_features_df = (
    fda_df[[
        "FDA_ProductID",
        "Therapeutic_Class",
        "EPC_Class",
        "MoA_Class",
        "DOSAGEFORMNAME",
        "ROUTENAME",
        "LABELERNAME",
        "MARKETINGCATEGORYNAME",
        "Strength",
        "ACTIVE_INGRED_UNIT",
        "Start_Marketing_Year"
    ]]
    .drop_duplicates(subset=["FDA_ProductID"])
    .copy()
)

cat_cols = [
    "Therapeutic_Class",
    "EPC_Class",
    "MoA_Class",
    "DOSAGEFORMNAME",
    "ROUTENAME",
    "LABELERNAME",
    "MARKETINGCATEGORYNAME",
    "ACTIVE_INGRED_UNIT"
]

for col in cat_cols:
    fda_features_df[col] = fda_features_df[col].fillna("Unknown")

print("FDA feature dataframe shape:", fda_features_df.shape)
print(fda_features_df.head(10))

Step 3: Creating FDA feature dataframe...
FDA feature dataframe shape: (111255, 11)
  FDA_ProductID                       Therapeutic_Class  \
0     000020152  G-Protein-linked Receptor Interactions   
1     000020213                                 Insulin   
2     000020243  G-Protein-linked Receptor Interactions   
3     000020800                                     nan   
4     000021152  G-Protein-linked Receptor Interactions   
5     000021200              Positron Emitting Activity   
6     000021214  G-Protein-linked Receptor Interactions   
7     000021220                                     nan   
8     000021243  G-Protein-linked Receptor Interactions   
9     000021340  G-Protein-linked Receptor Interactions   

                        EPC_Class                                MoA_Class  \
0         GLP-1 Receptor Agonist   G-Protein-linked Receptor Interactions    
1                        Insulin                                   Unknown   
2         GLP-1 Receptor Agonist

## MERGING both PRODUCT AND ORIGINAL DATASET

In [7]:
print("Step 4: Merging Medicaid with FDA features...")

df_merged = df.merge(
    fda_features_df,
    left_on="ProductID",
    right_on="FDA_ProductID",
    how="left"
)

print("Merged dataframe shape:", df_merged.shape)
print("FDA match count:", df_merged["FDA_ProductID"].notna().sum())
print("FDA match rate:", round(df_merged["FDA_ProductID"].notna().mean() * 100, 2), "%")

print(df_merged[[
    "NDC_standardized",
    "ProductID",
    "Therapeutic_Class",
    "EPC_Class",
    "MoA_Class",
    "DOSAGEFORMNAME",
    "ROUTENAME",
    "LABELERNAME",
    "MARKETINGCATEGORYNAME",
    "Strength",
    "ACTIVE_INGRED_UNIT",
    "Start_Marketing_Year"
]].head(10))

Step 4: Merging Medicaid with FDA features...
Merged dataframe shape: (7921315, 31)
FDA match count: 7421586
FDA match rate: 93.69 %
  NDC_standardized  ProductID                       Therapeutic_Class  \
0      00002143380  000021433                  GLP-1 Receptor Agonist   
1      00002143480  000021434                  GLP-1 Receptor Agonist   
2      00002143611  000021436                                     nan   
3      00002144509  000021445              Interleukin-17A Antagonist   
4      00002144511  000021445              Interleukin-17A Antagonist   
5      00002144527  000021445              Interleukin-17A Antagonist   
6      00002145780  000021457  G-Protein-linked Receptor Interactions   
7      00002146080  000021460  G-Protein-linked Receptor Interactions   
8      00002147180  000021471  G-Protein-linked Receptor Interactions   
9      00002148480  000021484  G-Protein-linked Receptor Interactions   

                     EPC_Class                                 

Dropping Unnecessary Features and Columns


In [8]:
print("Dropping unwanted columns, adding expansion-state flag, and creating target...")

# --------------------------------------------------
# 1. Drop unnecessary columns
# --------------------------------------------------
cols_to_drop = [
    "NDC",                  # raw NDC
    "Labeler Code",
    "Product Code",
    "Package Size",
    "FDA_ProductID",
    "MoA_Class",
    "NDC_clean",
    "NDC_match_flag",
    "DOSAGEFORMNAME",
    "ACTIVE_INGRED_UNIT",
    "Start_Marketing_Year"
]

df_model = df_merged.drop(
    columns=[col for col in cols_to_drop if col in df_merged.columns]
).copy()

# --------------------------------------------------
# 2. Add Medicaid expansion-state flag
# --------------------------------------------------
# States that adopted Medicaid expansion (including DC)
expansion_states = {
    "AK", "AZ", "AR", "CA", "CO", "CT", "DC", "DE", "HI", "IA", "ID",
    "IL", "IN", "KS", "KY", "LA", "MA", "MD", "ME", "MI", "MN", "MO",
    "MT", "NC", "ND", "NE", "NH", "NJ", "NM", "NV", "NY", "OH", "OK",
    "OR", "PA", "RI", "SD", "UT", "VA", "VT", "WA", "WV"
}

df_model["is_expansion_state"] = df_model["State"].astype(str).str.strip().isin(expansion_states)
df_model["is_expansion_state"] = df_model["is_expansion_state"].map({True: "Yes", False: "No"})

# --------------------------------------------------
# 3. Create target variable
# --------------------------------------------------
df_model = df_model[
    (df_model["Units Reimbursed"] > 0) &
    (df_model["Medicaid Amount Reimbursed"] > 0)
].copy()

df_model["cost_per_unit"] = (
    df_model["Medicaid Amount Reimbursed"] / df_model["Units Reimbursed"]
)

# Optional helpful engineered feature
df_model = df_model[df_model["Number of Prescriptions"] > 0].copy()
df_model["units_per_prescription"] = (
    df_model["Units Reimbursed"] / df_model["Number of Prescriptions"]
)

# --------------------------------------------------
# 4. Quick checks
# --------------------------------------------------
print("New shape:", df_model.shape)
print(df_model[[
    "State",
    "is_expansion_state",
    "Units Reimbursed",
    "Number of Prescriptions",
    "Medicaid Amount Reimbursed",
    "cost_per_unit",
    "units_per_prescription"
]].head(10))

Dropping unwanted columns, adding expansion-state flag, and creating target...
New shape: (3921418, 23)
   State is_expansion_state  Units Reimbursed  Number of Prescriptions  \
0     AK                Yes             226.0                    108.0   
1     AK                Yes             216.0                    108.0   
2     AK                Yes              32.0                     31.0   
4     AK                Yes              37.0                     36.0   
6     AK                Yes             178.0                     89.0   
7     AK                Yes             118.0                     59.0   
8     AK                Yes             152.0                     76.0   
9     AK                Yes             222.0                    111.0   
10    AK                Yes             226.0                    113.0   
11    AK                Yes             148.0                     74.0   

    Medicaid Amount Reimbursed  cost_per_unit  units_per_prescription  
0        

## Missing

In [9]:
print("Checking missing values...")

missing_counts = df_model.isna().sum()
missing_pct = (missing_counts / len(df_model)) * 100

missing_summary = pd.DataFrame({
    "Missing Count": missing_counts,
    "Missing %": missing_pct.round(2)
}).sort_values(by="Missing %", ascending=False)

print(missing_summary)

Checking missing values...
                                Missing Count  Missing %
Strength                               529160      13.49
MARKETINGCATEGORYNAME                  144153       3.68
EPC_Class                              144153       3.68
LABELERNAME                            144153       3.68
ROUTENAME                              144153       3.68
Therapeutic_Class                      144153       3.68
Utilization Type                            0       0.00
Units Reimbursed                            0       0.00
Product Name                                0       0.00
Suppression Used                            0       0.00
Quarter                                     0       0.00
Year                                        0       0.00
State                                       0       0.00
Medicaid Amount Reimbursed                  0       0.00
Number of Prescriptions                     0       0.00
NDC_standardized                            0       0.00
Prod

In [10]:
print("Dropping Strength and removing missing values...")

# ------------------------------------------
# 1. Drop Strength column
# ------------------------------------------
if "Strength" in df_model.columns:
    df_model = df_model.drop(columns=["Strength"])

# ------------------------------------------
# 2. Drop rows with ANY missing values
# ------------------------------------------
before_rows = len(df_model)

df_model = df_model.dropna()

after_rows = len(df_model)

print(f"Rows before: {before_rows}")
print(f"Rows after: {after_rows}")
print(f"Rows removed: {before_rows - after_rows}")

# ------------------------------------------
# 3. Final check
# ------------------------------------------
print("\nRemaining missing values:")
print(df_model.isna().sum())

print("\nPreview:")
print(df_model.head(10))

Dropping Strength and removing missing values...
Rows before: 3921418
Rows after: 3777265
Rows removed: 144153

Remaining missing values:
Utilization Type                  0
State                             0
Year                              0
Quarter                           0
Suppression Used                  0
Product Name                      0
Units Reimbursed                  0
Number of Prescriptions           0
Total Amount Reimbursed           0
Medicaid Amount Reimbursed        0
Non Medicaid Amount Reimbursed    0
Source_Year                       0
ProductID                         0
NDC_standardized                  0
Therapeutic_Class                 0
EPC_Class                         0
ROUTENAME                         0
LABELERNAME                       0
MARKETINGCATEGORYNAME             0
is_expansion_state                0
cost_per_unit                     0
units_per_prescription            0
dtype: int64

Preview:
   Utilization Type State  Year  Quarter  Suppr

In [ ]:
print("Creating medium-sized df_model.csv for dashboard...")

dashboard_cols = [
    "Year",
    "Quarter",
    "State",
    "ProductID",
    "Therapeutic_Class",
    "EPC_Class",
    "ROUTENAME",
    "LABELERNAME",
    "MARKETINGCATEGORYNAME",
    "Units Reimbursed",
    "Number of Prescriptions",
    "Medicaid Amount Reimbursed",
    "cost_per_unit",
    "units_per_prescription",
    "is_expansion_state"
]


dashboard_cols = [col for col in dashboard_cols if col in df_model.columns]

df_dashboard = df_model[dashboard_cols].copy()

# medium-sized sample
df_dashboard = df_dashboard.sample(
    n=min(150000, len(df_dashboard)),
    random_state=42
)


for col in df_dashboard.select_dtypes(include=["float"]).columns:
    df_dashboard[col] = df_dashboard[col].round(4)

# save
df_dashboard.to_csv("df_model.csv", index=False)

import os
file_size = os.path.getsize("df_model.csv") / (1024 * 1024)

print(f"df_model.csv saved successfully")
print(f"Shape: {df_dashboard.shape}")
print(f"File size: {file_size:.2f} MB")
print(df_dashboard.head())

Creating medium-sized df_model.csv for dashboard...
df_model.csv saved successfully
Shape: (150000, 15)
File size: 21.29 MB
         Year  Quarter State  ProductID  \
2224225  2024        2    MI  678770696   
5221259  2024        2    WV  007812020   
890818   2024        1    FL  633230113   
2260142  2024        1    MI  690970943   
5147717  2024        3    WI  724850670   

                                         Therapeutic_Class  \
2224225                                 Increased Diuresis   
5221259                     Penicillin-class Antibacterial   
890818                                       Antiprotozoal   
2260142  Decreased Central Nervous System Disorganized ...   
5147717                 Decreased Sebaceous Gland Activity   

                               EPC_Class                   ROUTENAME  \
2224225          Thiazide-like Diuretic                         ORAL   
5221259  Penicillin-class Antibacterial                         ORAL   
890818                    An

## Model Training

In [12]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

print("Preparing shared modeling dataset...")

df_compare = df_model.copy()

feature_cols = [
    "State",
    "Quarter",
    "Units Reimbursed",
    "Number of Prescriptions",
    "units_per_prescription",
    "Therapeutic_Class",
    "EPC_Class",
    "ROUTENAME",
    "LABELERNAME",
    "MARKETINGCATEGORYNAME",
    "is_expansion_state"
]

target_col = "cost_per_unit"

df_compare = df_compare[feature_cols + [target_col]].copy()

df_compare = df_compare[
    (df_compare[target_col] > 0) &
    np.isfinite(df_compare[target_col])
].copy()

# Trim extreme target outliers
lower = df_compare[target_col].quantile(0.01)
upper = df_compare[target_col].quantile(0.99)

df_compare = df_compare[
    (df_compare[target_col] >= lower) &
    (df_compare[target_col] <= upper)
].copy()

print("Shape after filtering:", df_compare.shape)

cat_cols = [
    "State",
    "Therapeutic_Class",
    "EPC_Class",
    "ROUTENAME",
    "LABELERNAME",
    "MARKETINGCATEGORYNAME",
    "is_expansion_state"
]

num_cols = [
    "Quarter",
    "Units Reimbursed",
    "Number of Prescriptions",
    "units_per_prescription"
]

X = df_compare[feature_cols].copy()
y = np.log1p(df_compare[target_col])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

results = []

def evaluate_model(model_name, y_true_log, y_pred_log):
    y_true = np.expm1(y_true_log)
    y_pred = np.expm1(y_pred_log)

    r2 = r2_score(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae = mean_absolute_error(y_true, y_pred)

    results.append({
        "Model": model_name,
        "R2": r2,
        "RMSE": rmse,
        "MAE": mae
    })

    print(f"{model_name}")
    print(f"R2:   {r2:.4f}")
    print(f"RMSE: {rmse:.4f}")
    print(f"MAE:  {mae:.4f}")
    print("-" * 50)

Preparing shared modeling dataset...
Shape after filtering: (3701719, 12)


In [ ]:
## Linear Regression

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression

preprocessor_lr = ColumnTransformer(
    transformers=[
        ("num", SimpleImputer(strategy="median"), num_cols),
        ("cat", Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(handle_unknown="ignore"))
        ]), cat_cols)
    ]
)

lr_pipeline = Pipeline([
    ("preprocessor", preprocessor_lr),
    ("model", LinearRegression())
])

print("Training Linear Regression...")
lr_pipeline.fit(X_train, y_train)

lr_pred_log = lr_pipeline.predict(X_test)
evaluate_model("Linear Regression", y_test, lr_pred_log)

Training Linear Regression...
Linear Regression
R2:   -0.0258
RMSE: 126.2800
MAE:  21.5898
--------------------------------------------------


## XGBoost

In [ ]:
import xgboost as xgb

X_train_xgb = X_train.copy()
X_test_xgb = X_test.copy()

for col in cat_cols:
    X_train_xgb[col] = X_train_xgb[col].astype("category")
    X_test_xgb[col] = X_test_xgb[col].astype("category")

print("Training XGBoost...")
xgb_model = xgb.XGBRegressor(
    objective="reg:squarederror",
    enable_categorical=True,
    tree_method="hist",
    n_estimators=300,
    learning_rate=0.05,
    max_depth=8,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1
)

xgb_model.fit(X_train_xgb, y_train)

xgb_pred_log = xgb_model.predict(X_test_xgb)
evaluate_model("XGBoost", y_test, xgb_pred_log)

Training XGBoost...
XGBoost
R2:   0.8329
RMSE: 50.9644
MAE:  6.4420
--------------------------------------------------


## LightGBM Model

In [13]:
import lightgbm as lgb

X_train_lgbm = X_train.copy()
X_test_lgbm = X_test.copy()

for col in cat_cols:
    X_train_lgbm[col] = X_train_lgbm[col].astype("category")
    X_test_lgbm[col] = X_test_lgbm[col].astype("category")

print("Training LightGBM...")
lgbm_model = lgb.LGBMRegressor(
    objective="regression",
    n_estimators=300,      # reduce from 2000
    learning_rate=0.05,
    num_leaves=20,         # smaller tree
    max_depth=8,           # limit depth
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1
)

lgbm_model.fit(
    X_train_lgbm,
    y_train,
    eval_set=[(X_test_lgbm, y_test)],
    eval_metric="rmse",
    categorical_feature=cat_cols,
    callbacks=[lgb.early_stopping(50), lgb.log_evaluation(100)]
)

lgbm_pred_log = lgbm_model.predict(X_test_lgbm, num_iteration=lgbm_model.best_iteration_)
evaluate_model("LightGBM", y_test, lgbm_pred_log)

Training LightGBM...
[LightGBM] [Warning] Categorical features with more bins than the configured maximum bin number found.
[LightGBM] [Warning] For categorical features, max_bin and max_bin_by_feature may be ignored with a large number of categories.
[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.165973 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1987
[LightGBM] [Info] Number of data points in the train set: 2961375, number of used features: 11
[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Start training from score 0.867746
Training until validation scores don't improve for 50 rounds
[100]	valid_0's rmse: 0.467625	valid_0's l2: 0.218673
[200]	valid_0's rmse: 0.436108	valid_0's l2: 0.19019
[300]	valid_

In [14]:
results_df = pd.DataFrame(results).sort_values("R2", ascending=False).reset_index(drop=True)
results_df

,Model,R2,RMSE,MAE
0,LightGBM,0.750517,62.276048,8.492365


In [15]:
## SAVING MODEL
import joblib

joblib.dump(lgbm_model, "lgbm_medicaid_model.pkl")
print("Saved LightGBM model as lgbm_medicaid_model.pkl")

Saved LightGBM model as lgbm_medicaid_model.pkl


In [ ]:
df_model.to_csv("df_model.csv", index=False)